# Latent tyre degradation: state-space walkthrough

This notebook demonstrates the model used by `latent_tyre_model.py`. It is a Kalman state estimator with a physically interpretable state and explicit uncertainty. It does **not** claim private tyre telemetry or an exact skewed-t likelihood.

## Model

For a lap-to-lap age increment $d$, $x=[D,r]^T$ contains latent fuel-corrected degradation $D$ (seconds) and its rate $r$ (seconds/lap).

$$x_{k+1}=\begin{bmatrix}1&d\\0&1\end{bmatrix}x_k+w_k,\qquad y_k=[1\;0]x_k+v_k.$$

The covariance $P$ moves through the same transition. A standard normal forecast gives $P(D>c)=1-\Phi((c-\mu)/\sigma)$ for cliff threshold $c$.

In [ ]:
import numpy as np
import pandas as pd
from latent_tyre_model import LatentTyreModel, first_cliff_forecast

# A deterministic illustrative stint. Replace with cleaned pipeline rows:
# columns must be TyreLife and Degradation_Delta.
life = np.arange(1, 19, dtype=float)
observed = np.maximum(0, .04 * (life - 2) + .007 * np.maximum(life - 11, 0) ** 2)
stint = pd.DataFrame({'TyreLife': life, 'Degradation_Delta': observed})
stint.head()

In [ ]:
# The prior rate and noise ordinarily come from fit_latent_tyre_model(train_df).
# Here they are deliberately stated so the demo is fully reproducible.
model = LatentTyreModel(prior_rate_s_per_lap=.045, observation_std_s=.16)
posterior = model.infer_stint(stint)
posterior[['TyreLife', 'Degradation_Delta', 'Prior_Degradation_Mean',
           'Latent_Degradation_Mean', 'Latent_Degradation_Std']].head()

In [ ]:
# Forecast from the final posterior. The filter robustly clips extreme
# innovations, so an anomalous slow lap does not dominate the state.
last = posterior.iloc[-1]
state = model.initial_state(last.TyreLife)
state.mean_s = last.Latent_Degradation_Mean
state.rate_s_per_lap = last.Latent_Degradation_Rate
state.covariance[0, 0] = last.Latent_Degradation_Std ** 2
forecast = model.forecast(state, np.arange(last.TyreLife, last.TyreLife + 16), cliff_threshold_s=1.5)
print('First tyre age at 80% cliff risk:', first_cliff_forecast(forecast, .80))
forecast.head()

## Validation protocol

Hold out whole races, fit `fit_latent_tyre_model` on remaining stints, and score `Prior_Degradation_Mean` before each held-out observation is assimilated. Plot actual fuel-corrected degradation against the predicted 70% and 90% bands. Report both held-out MAE and empirical interval coverage; neither alone establishes reliable pit-wall decisions.